# 3D City Skyline — Interactive Camera

A procedural city with hatched buildings, rendered as plotter-ready line art.
Use the sliders to orbit the camera around the skyline.

In [1]:
import numpy as np
from penpal import pen_width
from penpal.render3d import Camera, Scene, Mesh3D, TextureSpec, Wireframe

LW = pen_width(0.3)
LW_LIGHT = pen_width(0.2)

In [2]:
rng = np.random.default_rng(77)
scene = Scene()

grid_rows, grid_cols = 6, 8
block_size = 0.95

for row in range(grid_rows):
    for col in range(grid_cols):
        if rng.random() < 0.1:
            continue

        cx = (col - grid_cols / 2 + 0.5) * block_size
        cz = (row - grid_rows / 2 + 0.5) * block_size

        w = rng.uniform(0.35, 0.75)
        d = rng.uniform(0.35, 0.75)
        h = rng.exponential(1.0) + 0.3

        # Downtown cluster
        dist = np.sqrt(cx**2 + cz**2)
        if dist < 1.5:
            h *= rng.uniform(1.5, 2.5)
        elif dist < 3.0:
            h *= rng.uniform(0.8, 1.5)
        h = min(h, 5.0)

        angle = rng.choice([0, 30, 45, 60, 90, 135])
        spacing = rng.uniform(0.04, 0.1)

        if h > 3.0:
            spacing = rng.uniform(0.03, 0.06)
            style = rng.choice(['hatch', 'crosshatch'])
        elif h > 1.5:
            style = rng.choice(['hatch', 'hatch', 'crosshatch'])
        else:
            style = 'hatch'

        front_tex = TextureSpec(style=style, spacing=spacing, angle=angle)
        side_tex = TextureSpec(style=style, spacing=spacing, angle=(angle + 90) % 180)
        roof_tex = TextureSpec(style='crosshatch', spacing=rng.uniform(0.06, 0.12))

        scene.add(Mesh3D.box(
            size=(w, h, d),
            center=(cx, h / 2, cz),
            face_textures={
                'top': roof_tex,
                'front': front_tex, 'back': front_tex,
                'left': side_tex, 'right': side_tex,
            },
            face_layers={
                'top': 'roofs',
                'front': 'walls', 'back': 'walls',
                'left': 'walls', 'right': 'walls',
                'bottom': 'walls',
            },
        ))

scene.add(Mesh3D.plane(
    width=12, depth=10, center=(0, 0, 0), normal_axis='y',
    texture=TextureSpec(style='hatch', spacing=0.25, angle=90),
    layer='ground',
))

print(scene)

Scene(44 Mesh3D)


## Interactive camera

In [3]:
from ipywidgets import interact, FloatSlider
from IPython.display import display

@interact(
    azimuth=FloatSlider(min=0, max=360, step=5, value=22, description='Azimuth'),
    elevation=FloatSlider(min=-10, max=89, step=2, value=22, description='Elevation'),
    distance=FloatSlider(min=3, max=20, step=0.5, value=8, description='Distance'),
    fov=FloatSlider(min=20, max=120, step=5, value=55, description='FOV'),
)
def view(azimuth, elevation, distance, fov):
    cam = Camera.orbit(
        target=(0, 1.2, 0),
        distance=distance, azimuth=azimuth,
        elevation=elevation, fov=fov,
    )
    d = scene.render(cam, width=10, height=8)
    d.layer('walls', color='black', linewidth=LW)
    d.layer('roofs', color='black', linewidth=LW)
    d.layer('ground', color='#aaaaaa', linewidth=LW_LIGHT)
    display(d)

interactive(children=(FloatSlider(value=22.0, description='Azimuth', max=360.0, step=5.0), FloatSlider(value=2…